In [1]:
 !pip install -q openai-whisper jiwer


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 58.1 MB/s eta 0:00:00


In [2]:
import whisper

print("Whisper prêt")

Whisper prêt


In [3]:
# =========================================
# CELLULE 2
# IMPORTS
# =========================================

import whisper
import pandas as pd
import os
import re

from jiwer import wer

In [4]:
# =========================================
# CELLULE 3
# VERIFIER LES DATASETS KAGGLE
# =========================================

print(os.listdir("/kaggle/input/datasets/elkhamlichinada"))

['darijatranscritarabe', 'latindarija', 'audiodarija']


In [5]:
# =========================================
# CELLULE 4
# VERIFIER AUDIO
# =========================================

audio_dir = "/kaggle/input/datasets/elkhamlichinada/audiodarija/audio"

print(os.listdir(audio_dir)[:5])

['darija_04067.wav', 'darija_03751.wav', 'darija_10131.wav', 'darija_04334.wav', 'darija_01141.wav']


In [6]:
# =========================================
# CELLULE 5
# CHARGER METADATA ARABE
# =========================================

metadata = pd.read_csv(
    "/kaggle/input/datasets/elkhamlichinada/darijatranscritarabe/metadata_arabic.csv"
)

metadata.head()

,file,text
0,darija_00000.wav,فراسك أماما كون مهدي بقا ساكن معانا فالدار كون...
1,darija_00001.wav,أمزيان ملي كاتخرج بحال هاكا الطبالي والكراسا م...
2,darija_00002.wav,نتا لي بغيتي النهار لول قولتلك ماعندك ماتدير ب...
3,darija_00003.wav,تا حاجة ما بزاف عليك بالعكس نجيب شنو ما جبت غا...
4,darija_00004.wav,أنا راه غانبقا حاضيك غانسا راسي راه داير مع ال...


In [7]:
# =========================================
# CELLULE 6
# NETTOYAGE DATASET
# =========================================

# enlever lignes vides
metadata = metadata.dropna()

# reset index
metadata = metadata.reset_index(drop=True)

metadata.head()

,file,text
0,darija_00000.wav,فراسك أماما كون مهدي بقا ساكن معانا فالدار كون...
1,darija_00001.wav,أمزيان ملي كاتخرج بحال هاكا الطبالي والكراسا م...
2,darija_00002.wav,نتا لي بغيتي النهار لول قولتلك ماعندك ماتدير ب...
3,darija_00003.wav,تا حاجة ما بزاف عليك بالعكس نجيب شنو ما جبت غا...
4,darija_00004.wav,أنا راه غانبقا حاضيك غانسا راسي راه داير مع ال...


In [8]:
# =========================================
# CELLULE 7
# NORMALISATION ARABE
# =========================================

def normalize_arabic(text):

    text = str(text)

    # enlever ponctuation
    text = re.sub(r"[^\w\s]", " ", text)

    # enlever espaces multiples
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [9]:
# =========================================
# CELLULE 8
# CHARGER MODELE WHISPER
# =========================================

model = whisper.load_model("base")

print("MODELE CHARGE")

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 186MiB/s]


MODELE CHARGE


In [10]:
# =========================================
# CELLULE 9
# TEST SUR UN AUDIO
# =========================================

audio_file = metadata.iloc[0]["file"]

audio_path = os.path.join(audio_dir, audio_file)

print(audio_path)
print(os.path.exists(audio_path))

/kaggle/input/datasets/elkhamlichinada/audiodarija/audio/darija_00000.wav
True


In [11]:
# =========================================
# CELLULE 10
# TRANSCRIPTION WHISPER
# =========================================

result = model.transcribe(
    audio_path,
    language="ar"
)

prediction = result["text"]

print("===== PREDICTION =====")
print(prediction)

===== PREDICTION =====
 فرسك ممكوه من دي بقصة مع نظر كوش دي تبكش حالة من هامشة انشي لاني ما قدش كن عرف مقرام حتو ولكن كشرح لي كل شي


In [12]:
# =========================================
# CELLULE 11
# TEXTE REFERENCE
# =========================================

reference = metadata.iloc[0]["text"]

print("===== REFERENCE =====")
print(reference)

===== REFERENCE =====
فراسك أماما كون مهدي بقا ساكن معانا فالدار كون شديت الباك شحال هادي من نهار مشا عند الجيلالي مابقيتش كانعرف نقرا حيت هو لي كان كايشرحليا كلشي


In [13]:
# =========================================
# CELLULE 12
# CALCUL WER
# =========================================

error = wer(
    normalize_arabic(reference),
    normalize_arabic(prediction)
)

print("WER :", error)

WER : 0.9629629629629629


In [14]:
# =========================================
# CELLULE 13
# TEST PLUSIEURS MODELES
# =========================================

models = ["tiny", "base", "small", "medium", "large"]

results_all = {}

In [15]:
# =========================================
# CELLULE 14
# PETIT SUBSET TEST
# =========================================

subset = metadata.head(10)

subset

,file,text
0,darija_00000.wav,فراسك أماما كون مهدي بقا ساكن معانا فالدار كون...
1,darija_00001.wav,أمزيان ملي كاتخرج بحال هاكا الطبالي والكراسا م...
2,darija_00002.wav,نتا لي بغيتي النهار لول قولتلك ماعندك ماتدير ب...
3,darija_00003.wav,تا حاجة ما بزاف عليك بالعكس نجيب شنو ما جبت غا...
4,darija_00004.wav,أنا راه غانبقا حاضيك غانسا راسي راه داير مع ال...
5,darija_00005.wav,سي براهيم عندي ليك واحد الخبار غادا تفرحك هاد ...
6,darija_00006.wav,تا شي حاجة مابقات كيفما كانت خاص غير لي يفهم و...
7,darija_00007.wav,التجربة و الثقة ديالي فيها كبيرة ماعمرها خيباتني
8,darija_00008.wav,ان شاء الله علاش لا تا أنا ا بابا باغا نرجعليك...
9,darija_00009.wav,صحابي كاع تيضحكو عليا. تيقولو لي ولد السالمي ل...


In [16]:
# =========================================
# CELLULE 15
# BENCHMARK COMPLET
# =========================================

for model_name in models:

    print("\n======================")
    print("MODEL :", model_name)
    print("======================")

    model = whisper.load_model(model_name)

    wers = []

    for idx, row in subset.iterrows():

        audio_path = os.path.join(audio_dir, row["file"])

        reference = str(row["text"])

        try:

            result = model.transcribe(
                audio_path,
                language="ar"
            )

            prediction = result["text"]

            error = wer(
                normalize_arabic(reference),
                normalize_arabic(prediction)
            )

            wers.append(error)

            print("\n-------------------")
            print("FILE :", row["file"])

            print("\nREF :")
            print(reference)

            print("\nPRED :")
            print(prediction)

            print("\nWER :", round(error, 3))

        except Exception as e:

            print("ERREUR :", e)

    avg_wer = sum(wers) / len(wers)

    results_all[model_name] = avg_wer

    print("\nWER MOYEN :", round(avg_wer, 3))


MODEL : tiny


100%|██████████████████████████████████████| 72.1M/72.1M [00:00<00:00, 213MiB/s]



-------------------
FILE : darija_00000.wav

REF :
فراسك أماما كون مهدي بقا ساكن معانا فالدار كون شديت الباك شحال هادي من نهار مشا عند الجيلالي مابقيتش كانعرف نقرا حيت هو لي كان كايشرحليا كلشي

PRED :
 فرسةINDISTINCTة ليقزى مشروعونح من هذا مكلام در�� لذات檔 لعمس السام不知道

WER : 0.963

-------------------
FILE : darija_00001.wav

REF :
أمزيان ملي كاتخرج بحال هاكا الطبالي والكراسا مرا مرا كاتولي الدنيا عندك زاز لوز

PRED :
 اذهب تقوم با يلك الخرج پالا ال längت أق鳳 عن الإuishالura

WER : 1.0

-------------------
FILE : darija_00002.wav

REF :
نتا لي بغيتي النهار لول قولتلك ماعندك ماتدير بهاد الصداع

PRED :
 القوم!*

WER : 1.0

-------------------
FILE : darija_00003.wav

REF :
تا حاجة ما بزاف عليك بالعكس نجيب شنو ما جبت غايكون قليل فحقك

PRED :
 حي Nutال امر وزأفالك بلك ال؟اramento�ل أن خاتب سلك vaak بباق

WER : 1.0

-------------------
FILE : darija_00004.wav

REF :
أنا راه غانبقا حاضيك غانسا راسي راه داير مع السيد لي غايقادلي ليبيرجومون ديال السيت خاصني نمشي

PRED :
 ناول عالق Sew�י

WE

100%|████████████████████████████████████████| 461M/461M [00:01<00:00, 247MiB/s]



-------------------
FILE : darija_00000.wav

REF :
فراسك أماما كون مهدي بقا ساكن معانا فالدار كون شديت الباك شحال هادي من نهار مشا عند الجيلالي مابقيتش كانعرف نقرا حيت هو لي كان كايشرحليا كلشي

PRED :
 فرست كما ما كنتم بقصة معنا في الدار كنتم بقش حالة هذه منها مش عنجيلا اللي ما قدش كنا عرف نقرا حتى أولي كان كيشرح لي كل شيء

WER : 1.0

-------------------
FILE : darija_00001.wav

REF :
أمزيان ملي كاتخرج بحال هاكا الطبالي والكراسا مرا مرا كاتولي الدنيا عندك زاز لوز

PRED :
 عم زيام لي كتخرج بحلاككة طبالي وكراسة مرة مرة كشو لي دنيا عندك زاز نغس

WER : 0.929

-------------------
FILE : darija_00002.wav

REF :
نتا لي بغيتي النهار لول قولتلك ماعندك ماتدير بهاد الصداع

PRED :
 **ّادشز وال seine**

WER : 1.0

-------------------
FILE : darija_00003.wav

REF :
تا حاجة ما بزاف عليك بالعكس نجيب شنو ما جبت غايكون قليل فحقك

PRED :
 تحجم الزفالك بلاكس نجيب شنوما جبت ريكونا قليل في حقك

WER : 0.846

-------------------
FILE : darija_00004.wav

REF :
أنا راه غانبقا حاضيك غانسا راسي راه داير مع السيد

100%|██████████████████████████████████████| 1.42G/1.42G [00:09<00:00, 163MiB/s]



-------------------
FILE : darija_00000.wav

REF :
فراسك أماما كون مهدي بقا ساكن معانا فالدار كون شديت الباك شحال هادي من نهار مشا عند الجيلالي مابقيتش كانعرف نقرا حيت هو لي كان كايشرحليا كلشي

PRED :
 فرس كما ما كن مهند بقصة معنا في الدار كنشديت باكش حالة هادى منهار مش عان جيلال لما بقتش كن عرف نقرأ حتى ولي كان كيش رح ليه كل شي

WER : 1.074

-------------------
FILE : darija_00001.wav

REF :
أمزيان ملي كاتخرج بحال هاكا الطبالي والكراسا مرا مرا كاتولي الدنيا عندك زاز لوز

PRED :
 أمزي أملي كنت خرج بعلاك قطب عليك وكراسة مرة مرة حتقول الدنيا عندك زازلوز

WER : 0.929

-------------------
FILE : darija_00002.wav

REF :
نتا لي بغيتي النهار لول قولتلك ماعندك ماتدير بهاد الصداع

PRED :
 ن deutschen في

WER : 1.0

-------------------
FILE : darija_00003.wav

REF :
تا حاجة ما بزاف عليك بالعكس نجيب شنو ما جبت غايكون قليل فحقك

PRED :
 اتحجم الزفالك بالاكس نجيب شنو ما جبتوه يكون قليل في حقك

WER : 0.769

-------------------
FILE : darija_00004.wav

REF :
أنا راه غانبقا حاضيك غانسا راسي راه داير 

100%|█████████████████████████████████████| 2.88G/2.88G [00:42<00:00, 72.0MiB/s]



-------------------
FILE : darija_00000.wav

REF :
فراسك أماما كون مهدي بقا ساكن معانا فالدار كون شديت الباك شحال هادي من نهار مشا عند الجيلالي مابقيتش كانعرف نقرا حيت هو لي كان كايشرحليا كلشي

PRED :
 فرس كمام ما كن مهندي بقصة معانا في الدار كنش دي تباكش حالها هذي من هار مش عين جيلان لي ما بقتش كن عارف نقرا حتوالي كان كنشرح لي كل شي

WER : 1.037

-------------------
FILE : darija_00001.wav

REF :
أمزيان ملي كاتخرج بحال هاكا الطبالي والكراسا مرا مرا كاتولي الدنيا عندك زاز لوز

PRED :
 أمس يامني كتخرج بحلاقة طبالية وكراسة مرة مرة حتول الدنيا عندك زاز نوكس

WER : 0.786

-------------------
FILE : darija_00002.wav

REF :
نتا لي بغيتي النهار لول قولتلك ماعندك ماتدير بهاد الصداع

PRED :
 نتالي بريتي نهار الولد الملك من 14

WER : 1.0

-------------------
FILE : darija_00003.wav

REF :
تا حاجة ما بزاف عليك بالعكس نجيب شنو ما جبت غايكون قليل فحقك

PRED :
 تحاجة مزيفة لك بالعكس نجيب شنو ما جبت ريكون قليل في حقك

WER : 0.615

-------------------
FILE : darija_00004.wav

REF :
أنا راه غانبقا حاض

In [17]:
# =========================================
# CELLULE 16
# RESULTATS FINAUX
# =========================================

print("\n======================")
print("RESULTATS FINAUX")
print("======================")

for model_name, score in results_all.items():

    print(f"{model_name} ---> {score:.3f}")


RESULTATS FINAUX
tiny ---> 1.017
base ---> 0.953
small ---> 0.913
medium ---> 0.928
large ---> 0.853


In [18]:
# =========================================
# CELLULE 17
# MEILLEUR MODELE
# =========================================

best_model = min(results_all, key=results_all.get)

print("MEILLEUR MODELE :", best_model)
print("WER :", results_all[best_model])

MEILLEUR MODELE : large
WER : 0.8527024827024826


In [19]:
from jiwer import wer
import whisper
import pandas as pd
import os
import re

# =========================
# NORMALISATION (SAFE)
# =========================
def normalize(text):

    text = str(text).lower()

    # juste nettoyage léger (PAS de substitution phonétique)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


# =========================
# LOAD DATASET
# =========================
latin_df = pd.read_csv(
    "/kaggle/input/datasets/elkhamlichinada/latindarija/metadata.csv",
    sep=";"
)

latin_df.columns = latin_df.columns.str.strip()
latin_df = latin_df.iloc[1:].reset_index(drop=True)

subset = latin_df.head(10)

audio_dir = "/kaggle/input/datasets/elkhamlichinada/audiodarija/audio"


# =========================
# MODELS
# =========================
models = ["tiny", "base", "small", "medium"]  # skip large for test

results_all = {}


# =========================
# LOOP MODELS
# =========================
for model_name in models:

    print("\n======================")
    print("MODEL :", model_name)
    print("======================")

    model = whisper.load_model(model_name)

    wers = []

    for idx, row in subset.iterrows():

        audio_path = os.path.join(audio_dir, row["file"])
        reference = str(row["text"])

        if not os.path.exists(audio_path):
            print("Missing file:", audio_path)
            continue

        try:

            result = model.transcribe(audio_path)
            prediction = result["text"]

            error = wer(
                normalize(reference),
                normalize(prediction)
            )

            wers.append(error)

            print("\nFILE :", row["file"])
            print("WER :", round(error, 3))

        except Exception as e:
            print("ERROR :", e)

    if len(wers) > 0:
        avg_wer = sum(wers) / len(wers)
    else:
        avg_wer = float("inf")

    results_all[model_name] = avg_wer

    print("\nWER MOYEN :", round(avg_wer, 3))


# =========================
# FINAL RESULTS
# =========================
print("\n======================")
print("RESULTATS FINAUX")
print("======================")

for model_name, score in results_all.items():
    print(model_name, "--->", round(score, 3))

best_model = min(results_all, key=results_all.get)

print("\nMEILLEUR MODELE :", best_model)
print("WER :", results_all[best_model])


MODEL : tiny

FILE : darija_00000.wav
WER : 1.0

FILE : darija_00001.wav
WER : 1.0

FILE : darija_00002.wav
WER : 1.0

FILE : darija_00003.wav
WER : 1.0

FILE : darija_00004.wav
WER : 1.0

FILE : darija_00005.wav
WER : 1.0

FILE : darija_00006.wav
WER : 1.0

FILE : darija_00007.wav
WER : 1.0

FILE : darija_00008.wav
WER : 1.0

FILE : darija_00009.wav
WER : 1.0

WER MOYEN : 1.0

MODEL : base

FILE : darija_00000.wav
WER : 1.0

FILE : darija_00001.wav
WER : 1.0

FILE : darija_00002.wav
WER : 1.0

FILE : darija_00003.wav
WER : 1.0

FILE : darija_00004.wav
WER : 1.0

FILE : darija_00005.wav
WER : 1.091

FILE : darija_00006.wav
WER : 1.125

FILE : darija_00007.wav
WER : 1.125

FILE : darija_00008.wav
WER : 1.412

FILE : darija_00009.wav
WER : 1.0

WER MOYEN : 1.075

MODEL : small

FILE : darija_00000.wav
WER : 1.037

FILE : darija_00001.wav
WER : 1.0

FILE : darija_00002.wav
WER : 1.0

FILE : darija_00003.wav
WER : 1.0

FILE : darija_00004.wav
WER : 1.0

FILE : darija_00005.wav
WER : 1.0



In [20]:
import pandas as pd
from datasets import Dataset, Audio

# =========================
# LOAD CSV
# =========================

df = pd.read_csv(
    "/kaggle/input/datasets/elkhamlichinada/darijatranscritarabe/metadata_arabic.csv",
    sep=","
)

df.columns = df.columns.str.strip()

# enlever première ligne exemple
df = df.iloc[1:].reset_index(drop=True)

# =========================
# AUDIO PATH
# =========================

audio_dir = "/kaggle/input/datasets/elkhamlichinada/audiodarija"

df["audio"] = df["file"].apply(
    lambda x: f"{audio_dir}/{x}"
)

# =========================
# TEXT COLUMN
# =========================

df = df.rename(columns={
    "text": "sentence"
})

df = df[["audio", "sentence"]]

print(df.head())

                                               audio  \
0  /kaggle/input/datasets/elkhamlichinada/audioda...   
1  /kaggle/input/datasets/elkhamlichinada/audioda...   
2  /kaggle/input/datasets/elkhamlichinada/audioda...   
3  /kaggle/input/datasets/elkhamlichinada/audioda...   
4  /kaggle/input/datasets/elkhamlichinada/audioda...   

                                            sentence  
0  أمزيان ملي كاتخرج بحال هاكا الطبالي والكراسا م...  
1  نتا لي بغيتي النهار لول قولتلك ماعندك ماتدير ب...  
2  تا حاجة ما بزاف عليك بالعكس نجيب شنو ما جبت غا...  
3  أنا راه غانبقا حاضيك غانسا راسي راه داير مع ال...  
4  سي براهيم عندي ليك واحد الخبار غادا تفرحك هاد ...  


In [21]:
# convertir en dataset HuggingFace
dataset = Dataset.from_pandas(df)

# audio
dataset = dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

print(dataset)

Dataset({
    features: ['audio', 'sentence'],
    num_rows: 10444
})


In [22]:
# train / test split

dataset = dataset.train_test_split(
    test_size=0.1,
    seed=42
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 9399
    })
    test: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 1045
    })
})


In [23]:
from transformers import WhisperProcessor

models = [
    "openai/whisper-tiny",
    "openai/whisper-base",
    "openai/whisper-small",
    "openai/whisper-medium"
]

processors = {}

for model_name in models:

    processor = WhisperProcessor.from_pretrained(
        model_name,
        language="Arabic",
        task="transcribe"
    )

    processors[model_name] = processor

    print("Loaded :", model_name)

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Loaded : openai/whisper-medium


In [24]:
from transformers import WhisperFeatureExtractor
from transformers import WhisperTokenizer
from transformers import WhisperProcessor

model_name = "openai/whisper-small"

feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name)

tokenizer = WhisperTokenizer.from_pretrained(
    model_name,
    language="Arabic",
    task="transcribe"
)

processor = WhisperProcessor.from_pretrained(
    model_name,
    language="Arabic",
    task="transcribe"
)

In [25]:
!pip install -q transformers datasets accelerate evaluate jiwer torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


In [26]:
import os
import pandas as pd
import torchaudio
from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import evaluate

In [27]:
df = pd.read_csv(
    "/kaggle/input/datasets/elkhamlichinada/darijatranscritarabe/metadata_arabic.csv",
    sep=","
)

df.columns = df.columns.str.strip()
df = df.iloc[1:].reset_index(drop=True)

audio_dir = "/kaggle/input/datasets/elkhamlichinada/audiodarija/audio"

In [28]:
def build_path(x):
    return os.path.join(audio_dir, x)

df["audio"] = df["file"].apply(build_path)
df = df.rename(columns={"text": "sentence"})

# garder uniquement fichiers existants
df = df[df["audio"].apply(os.path.exists)]

df = df[["audio", "sentence"]]

In [29]:
dataset = Dataset.from_pandas(df)

dataset = dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

dataset = dataset.train_test_split(test_size=0.1, seed=42)

In [30]:
models = [
    "openai/whisper-tiny",
    "openai/whisper-base"
]

In [31]:
import os
import pandas as pd
from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import evaluate

In [32]:
df = pd.read_csv(
    "/kaggle/input/datasets/elkhamlichinada/darijatranscritarabe/metadata_arabic.csv",
    sep=","
)

df.columns = df.columns.str.strip()
df = df.iloc[1:].reset_index(drop=True)

audio_dir = "/kaggle/input/datasets/elkhamlichinada/audiodarija/audio"

df["audio"] = df["file"].apply(lambda x: os.path.join(audio_dir, x))
df = df.rename(columns={"text": "sentence"})

# garder seulement fichiers existants
df = df[df["audio"].apply(os.path.exists)]

df = df[["audio", "sentence"]]

In [33]:
import os
import gc
import torch
import evaluate
import numpy as np
import pandas as pd

from datasets import Dataset, Audio, concatenate_datasets, load_from_disk
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback
)

# =========================
# CONFIG
# =========================
MODEL_NAME = "openai/whisper-small"
AUDIO_DIR  = "/kaggle/input/datasets/elkhamlichinada/audiodarija/audio"
OUTPUT_DIR = "./whisper-small-darija-finetuned"
CACHE_DIR  = "./preprocessed_cache"
N_SAMPLES  = 5000   # ← change ici : None = tout le dataset, ou un nombre ex: 2000

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# WER METRIC
# =========================
wer_metric = evaluate.load("wer")

# =========================
# PRÉPARER LE DATAFRAME
# (df est déjà chargé dans la cellule précédente)
# =========================
df_work = df.copy()

if N_SAMPLES is not None and N_SAMPLES < len(df_work):
    df_work = df_work.sample(n=N_SAMPLES, random_state=42).reset_index(drop=True)
    print(f"✅ Dataset réduit à {N_SAMPLES} samples")
else:
    print(f"✅ Dataset complet : {len(df_work)} samples")

# =========================
# CRÉER LE DATASET HF
# =========================
dataset = Dataset.from_pandas(df_work)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(f"   Train : {len(dataset['train'])} samples")
print(f"   Test  : {len(dataset['test'])} samples")

# =========================
# PROCESSOR + MODEL
# =========================
processor = WhisperProcessor.from_pretrained(
    MODEL_NAME, language="arabic", task="transcribe"
)

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.config.use_cache = False
model.gradient_checkpointing_enable()

model.generation_config.forced_decoder_ids    = None
model.generation_config.suppress_tokens       = None
model.generation_config.begin_suppress_tokens = None
model.generation_config.language              = "arabic"
model.generation_config.task                  = "transcribe"

# =========================
# WER BASELINE (avant fine-tuning)
# =========================
print("\n📊 Calcul du WER baseline (avant fine-tuning)...")
model.eval()
baseline_preds, baseline_refs = [], []
n_baseline = min(50, len(dataset["test"]))
sample_test = dataset["test"].select(range(n_baseline))

for item in sample_test:
    inputs = processor.feature_extractor(
        item["audio"]["array"], sampling_rate=16000, return_tensors="pt"
    )
    with torch.no_grad():
        predicted_ids = model.generate(inputs["input_features"])
    pred = processor.tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    baseline_preds.append(pred)
    baseline_refs.append(item["sentence"])

baseline_wer = wer_metric.compute(predictions=baseline_preds, references=baseline_refs)
print(f"📊 WER baseline : {baseline_wer:.3f}")

# =========================
# PREPROCESSING PAR CHUNKS (évite OverflowError PyArrow >2GB)
# =========================
def preprocess_dataset_chunked(ds, processor, split_name, chunk_size=500):
    total       = len(ds)
    chunk_paths = []

    print(f"  Preprocessing {total} samples ({split_name}) par chunks de {chunk_size}...")

    for start in range(0, total, chunk_size):
        end        = min(start + chunk_size, total)
        chunk_path = os.path.join(CACHE_DIR, f"{split_name}_chunk_{start}_{end}")

        if os.path.exists(chunk_path):
            print(f"  [{start}-{end}] chunk déjà en cache ✅")
            chunk_paths.append(chunk_path)
            continue

        input_features, labels = [], []
        for i in range(start, end):
            item = ds[i]
            feat = processor.feature_extractor(
                item["audio"]["array"], sampling_rate=16000
            ).input_features[0]
            lbl  = processor.tokenizer(item["sentence"]).input_ids
            input_features.append(feat)
            labels.append(lbl)

        chunk_ds = Dataset.from_dict({
            "input_features": input_features,
            "labels": labels
        })
        chunk_ds.save_to_disk(chunk_path)
        chunk_paths.append(chunk_path)
        print(f"  [{start}-{end}] sauvegardé ✅")

        del input_features, labels, chunk_ds
        gc.collect()

    print(f"  Assemblage des {len(chunk_paths)} chunks...")
    all_chunks = [load_from_disk(p) for p in chunk_paths]
    full_ds    = concatenate_datasets(all_chunks)
    print(f"  ✅ Dataset final : {len(full_ds)} samples")
    return full_ds

print("\nPreprocessing train set...")
train_ds = preprocess_dataset_chunked(dataset["train"], processor, "train", chunk_size=500)

print("\nPreprocessing test set...")
test_ds  = preprocess_dataset_chunked(dataset["test"],  processor, "test",  chunk_size=500)

# =========================
# DATA COLLATOR
# =========================
class DataCollatorSpeechSeq2SeqWithPadding:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )
        batch["input_features"] = batch["input_features"].to(torch.float32)

        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"] != 1, -100
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

# =========================
# COMPUTE METRICS
# =========================
def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]
    pred_ids = np.array(pred_ids)
    if pred_ids.ndim == 3 or pred_ids.dtype in [np.float32, np.float64]:
        pred_ids = np.argmax(pred_ids, axis=-1)
    label_ids = np.array(label_ids)
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": wer_metric.compute(predictions=pred_str, references=label_str)}

# =========================
# TRAINING ARGS
# =========================
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=200,
    num_train_epochs=10,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    logging_steps=50,
    report_to="none",
    remove_unused_columns=False,
    predict_with_generate=True,
    generation_max_length=225,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# =========================
# FINE-TUNING
# =========================
print(f"\n🚀 Début du fine-tuning sur {len(train_ds)} samples...")
print(f"   Epochs max    : 10 (early stopping patience=3)")
print(f"   Learning rate : 1e-5")
print(f"   Batch size    : 2 × 2 GPUs = 4 effectif")
print()

trainer.train()

# =========================
# RÉSULTATS FINAUX
# =========================
eval_result  = trainer.evaluate()
final_wer    = eval_result["eval_wer"]
amelioration = (baseline_wer - final_wer) / baseline_wer * 100

print("\n" + "="*40)
print("  RÉSULTATS FINE-TUNING whisper-small")
print("="*40)
print(f"  📊 WER avant  : {baseline_wer:.3f}")
print(f"  📊 WER après  : {final_wer:.3f}")
print(f"  📈 Amélioration : {amelioration:.1f}%")
print("="*40)

# =========================
# SAUVEGARDE
# =========================
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"\n✅ Modèle sauvegardé dans {OUTPUT_DIR}")
print("   → Va dans Output → télécharge le dossier pour le réutiliser")

✅ Dataset réduit à 5000 samples
   Train : 4500 samples
   Test  : 500 samples


model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]


📊 Calcul du WER baseline (avant fine-tuning)...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



  RÉSULTATS FINE-TUNING whisper-small
  📊 WER avant  : 1.334
  📊 WER après  : 0.458
  📈 Amélioration : 65.7%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Modèle sauvegardé dans ./whisper-small-darija-finetuned
   → Va dans Output → télécharge le dossier pour le réutiliser
